# Notebook 1 — Prepare the Restricted-Dog Image Dataset

This notebook downloads, extracts, validates, and splits the Stanford Dogs images into a binary computer-vision dataset:

- `restricted`
- `unrestricted`

It performs file and folder work, so it should run on **CPU**. The finished dataset is saved persistently in Google Drive under:

```text
/content/drive/MyDrive/restricted-dog-cnn
```


In [ ]:
# ============================================================
# 0. Environment check: Notebook 1 should run in COLAB on CPU
# ============================================================
import os
from pathlib import Path

try:
    import tensorflow as tf
    gpus = tf.config.list_physical_devices("GPU")
except Exception:
    gpus = []

print("=" * 70)
print("NOTEBOOK 1 ENVIRONMENT CHECK")
print("=" * 70)

if gpus:
    print("WARNING: A GPU is currently attached.")
    print("Notebook 1 does not need a GPU. It is download/extract/split work.")
    print("Recommended: Runtime -> Change runtime type -> CPU before running this notebook.")
else:
    print("Correct: no GPU detected. CPU runtime is appropriate for Notebook 1.")

print("\nCurrent working directory:", Path.cwd())


NOTEBOOK 1 ENVIRONMENT CHECK
Correct: no GPU detected. CPU runtime is appropriate for Notebook 1.

Current working directory: /content


In [ ]:
# ============================================================
# 1. Mount Google Drive
# ============================================================
from google.colab import drive
drive.mount("/content/drive")

print("Drive mounted.")


Mounted at /content/drive
Drive mounted.


In [ ]:
# ============================================================
# 2. Project paths
# ============================================================
from pathlib import Path


PROJECT_ROOT = Path("/content/drive/MyDrive") / PROJECT_FOLDER_NAME
DRIVE_DATA = PROJECT_ROOT / "data"

# Fast temporary storage on the Colab machine
LOCAL_ROOT = Path("/content/restricted_dogs_cnn")
LOCAL_RAW = LOCAL_ROOT / "raw"
LOCAL_SPLIT = LOCAL_ROOT / "split"
LOCAL_TAR = LOCAL_ROOT / "images.tar"

# Persistent files and folders in Google Drive
DRIVE_TAR = DRIVE_DATA / "images.tar"
DRIVE_TRAIN = DRIVE_DATA / "train"
DRIVE_VAL = DRIVE_DATA / "val"
DRIVE_TEST = DRIVE_DATA / "test"

for folder in [
    PROJECT_ROOT,
    DRIVE_DATA,
    LOCAL_ROOT,
    LOCAL_RAW,
    LOCAL_SPLIT,
]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Drive data:  ", DRIVE_DATA)
print("Local work:  ", LOCAL_ROOT)


Project root: /content/drive/MyDrive/restricted-dog-computer-vision
Drive data:   /content/drive/MyDrive/restricted-dog-computer-vision/data
Local work:   /content/restricted_dogs_work


In [ ]:
# ============================================================
# 3. Configuration
# ============================================================
SEED = 58
N_UNRESTRICTED = 24
SPLIT = (0.60, 0.20, 0.20)

# Set True to delete and rebuild Drive train/val/test folders.
REBUILD_SPLIT = False

# If True, copy/download/extract raw data again locally.
# Usually leave True because /content is temporary and fast.
REBUILD_LOCAL_RAW = True

TAR_URL = "http://vision.stanford.edu/aditya86/ImageNetDogs/images.tar"

CLASS_NAMES = ["unrestricted", "restricted"]

RESTRICTED_BREEDS = [
    "Bull Mastiff",
    "Doberman Pinscher",
    "German Shepherd",
    "Rhodesian Ridgeback",
    "Rottweiler",
    "Staffordshire Bull Terrier",
]

MANUAL_OVERRIDES = {
    # Example:
    # "Doberman Pinscher": "n02107142-Doberman",
}

print("Seed:", SEED)
print("Restricted breeds:", len(RESTRICTED_BREEDS))
for breed in RESTRICTED_BREEDS:
    print(" -", breed)
print("Unrestricted breeds to sample:", N_UNRESTRICTED)
print("Split:", SPLIT)
print("Rebuild Drive split:", REBUILD_SPLIT)


Seed: 58
Restricted breeds: 6
 - Bull Mastiff
 - Doberman Pinscher
 - German Shepherd
 - Rhodesian Ridgeback
 - Rottweiler
 - Staffordshire Bull Terrier
Unrestricted breeds to sample: 24
Split: (0.6, 0.2, 0.2)
Rebuild Drive split: False


In [ ]:
# ============================================================
# 4. Idempotency check for finished Drive split
# ============================================================
import shutil

def count_images(folder: Path) -> int:
    if not folder.exists():
        return 0
    return sum(1 for p in folder.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"})

def split_exists() -> bool:
    required = [
        DRIVE_TRAIN / "restricted",
        DRIVE_TRAIN / "unrestricted",
        DRIVE_VAL / "restricted",
        DRIVE_VAL / "unrestricted",
        DRIVE_TEST / "restricted",
        DRIVE_TEST / "unrestricted",
    ]
    return all(p.exists() and count_images(p) > 0 for p in required)

if REBUILD_SPLIT:
    print("REBUILD_SPLIT=True: removing old Drive split folders.")
    for folder in [DRIVE_TRAIN, DRIVE_VAL, DRIVE_TEST]:
        if folder.exists():
            shutil.rmtree(folder)

if split_exists():
    print("Finished Drive split already exists. Raw extraction will be skipped.")
    SKIP_BUILD = True
else:
    print("Finished Drive split missing or incomplete. Building from raw data.")
    SKIP_BUILD = False


Finished Drive split missing or incomplete. Building from raw data.


In [ ]:
# ============================================================
# 5. Prepare local tar file
# ============================================================
import urllib.request
import shutil
from pathlib import Path

def file_size_gb(path: Path) -> float:
    return path.stat().st_size / (1024 ** 3)

if not SKIP_BUILD:
    if REBUILD_LOCAL_RAW and LOCAL_ROOT.exists():
        # Keep LOCAL_ROOT itself but clean raw/split/tar from previous attempt
        for item in [LOCAL_RAW, LOCAL_SPLIT, LOCAL_TAR]:
            if item.exists():
                if item.is_dir():
                    shutil.rmtree(item)
                else:
                    item.unlink()
        LOCAL_RAW.mkdir(parents=True, exist_ok=True)
        LOCAL_SPLIT.mkdir(parents=True, exist_ok=True)

    if DRIVE_TAR.exists() and DRIVE_TAR.stat().st_size > 700_000_000:
        print("Copying existing tar from Drive to fast local disk.")
        print("From:", DRIVE_TAR)
        print("To:  ", LOCAL_TAR)
        shutil.copy2(DRIVE_TAR, LOCAL_TAR)
    else:
        print("Drive tar missing or too small. Downloading directly to local disk.")
        urllib.request.urlretrieve(TAR_URL, LOCAL_TAR)
        print("Downloaded:", LOCAL_TAR)

        print("Saving a persistent copy of images.tar to Drive for future runs.")
        shutil.copy2(LOCAL_TAR, DRIVE_TAR)

    print("Local tar exists:", LOCAL_TAR.exists())
    print("Local tar size: ", round(file_size_gb(LOCAL_TAR), 3), "GB")

    if LOCAL_TAR.stat().st_size < 700_000_000:
        raise RuntimeError("images.tar is too small. Delete it and rerun the download cell.")
else:
    print("Skipped: finished split already exists.")


Drive tar missing or too small. Downloading directly to local disk.
Downloaded: /content/restricted_dogs_work/images.tar
Saving a persistent copy of images.tar to Drive for future runs.
Local tar exists: True
Local tar size:  0.739 GB


In [ ]:
# ============================================================
# 6. Extract Stanford Dogs locally in /content
# ============================================================
import tarfile
import time

if not SKIP_BUILD:
    print("Extracting locally. This should be much faster than extracting to Drive.")
    start = time.time()

    with tarfile.open(LOCAL_TAR) as tar:
        tar.extractall(LOCAL_RAW, filter="data")

    elapsed = (time.time() - start) / 60
    print(f"Extraction complete in {elapsed:.1f} minutes.")

    LOCAL_IMAGES = LOCAL_RAW / "Images"
    if not LOCAL_IMAGES.exists():
        raise FileNotFoundError(f"Expected folder not found: {LOCAL_IMAGES}")

    folders = sorted([p for p in LOCAL_IMAGES.iterdir() if p.is_dir()])
    print("Breed folders found:", len(folders))
    print("First five folders:", [p.name for p in folders[:5]])

    if len(folders) < 100:
        raise RuntimeError("Too few breed folders found. Extraction may be incomplete.")
else:
    print("Skipped: finished split already exists.")


Extracting locally. This should be much faster than extracting to Drive.
Extraction complete in 0.3 minutes.
Breed folders found: 120
First five folders: ['n02085620-Chihuahua', 'n02085782-Japanese_spaniel', 'n02085936-Maltese_dog', 'n02086079-Pekinese', 'n02086240-Shih-Tzu']


In [ ]:
# ============================================================
# 7. Match restricted breeds to Stanford folders
# ============================================================
import re
from difflib import SequenceMatcher

GENERIC_TOKENS = {
    "bull", "dog", "terrier", "hound", "spaniel", "setter",
    "retriever", "pointer", "shepherd", "mastiff", "pinscher",
    "poodle", "collie", "husky"
}

def normalise_name(s: str) -> str:
    s = re.sub(r"^n\d+[-_]", "", s)
    s = re.sub(r"[-_]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip().lower()
    return s

def tokenise_name(s: str) -> set:
    return set(normalise_name(s).split())

def fuzzy_match(target: str, candidates: list[str]):
    target_norm = normalise_name(target)
    target_tokens = tokenise_name(target)
    target_squash = "".join(sorted(target_tokens))
    distinctive = {
        t for t in target_tokens
        if t not in GENERIC_TOKENS and len(t) >= 5
    }

    best = None
    best_score = -1
    best_reason = ""

    for cand in candidates:
        cand_norm = normalise_name(cand)
        cand_tokens = tokenise_name(cand)
        cand_concat = "".join(cand_tokens)
        cand_squash = "".join(sorted(cand_tokens))

        s1 = SequenceMatcher(None, target_norm, cand_norm).ratio()
        s2 = SequenceMatcher(None, target_squash, cand_squash).ratio()
        distinctive_hit = any(t in cand_concat for t in distinctive) if distinctive else False
        all_present = all(t in cand_concat for t in target_tokens)

        score = max(s1, s2)
        if distinctive_hit:
            score += 0.40
        if all_present:
            score += 0.30

        if score > best_score:
            best = cand
            best_score = score
            best_reason = (
                f"ratio_norm={s1:.2f}, ratio_squash={s2:.2f}, "
                f"distinctive_hit={distinctive_hit}, all_tokens={all_present}"
            )

    return best, best_score, best_reason

if not SKIP_BUILD:
    all_folders = sorted([p.name for p in LOCAL_IMAGES.iterdir() if p.is_dir()])
    restricted_folders = []
    threshold = 0.85

    print("Matching restricted breeds:")
    print("-" * 70)

    for breed in RESTRICTED_BREEDS:
        if breed in MANUAL_OVERRIDES:
            match = MANUAL_OVERRIDES[breed]
            if match not in all_folders:
                raise RuntimeError(f"Manual override for {breed} not found: {match}")
            restricted_folders.append(match)
            print(f"{breed:30s} -> {match} (manual override)")
            continue

        match, score, reason = fuzzy_match(breed, all_folders)
        print(f"{breed:30s} -> {match}")
        print(f"  score={score:.2f}; {reason}")

        if score < threshold:
            raise RuntimeError(
                f"Low-confidence match for {breed}. "
                f"Add a MANUAL_OVERRIDES entry and rerun."
            )
        restricted_folders.append(match)

    restricted_folders = sorted(set(restricted_folders))
    print("-" * 70)
    print("Restricted Stanford folders:", len(restricted_folders))
    for folder in restricted_folders:
        print(" -", folder)
else:
    print("Skipped: finished split already exists.")


Matching restricted breeds:
----------------------------------------------------------------------
Bull Mastiff                   -> n02108422-bull_mastiff
  score=1.30; ratio_norm=1.00, ratio_squash=1.00, distinctive_hit=False, all_tokens=True
Doberman Pinscher              -> n02107142-Doberman
  score=1.07; ratio_norm=0.64, ratio_squash=0.67, distinctive_hit=True, all_tokens=False
German Shepherd                -> n02106662-German_shepherd
  score=1.70; ratio_norm=1.00, ratio_squash=1.00, distinctive_hit=True, all_tokens=True
Rhodesian Ridgeback            -> n02087394-Rhodesian_ridgeback
  score=1.70; ratio_norm=1.00, ratio_squash=1.00, distinctive_hit=True, all_tokens=True
Rottweiler                     -> n02106550-Rottweiler
  score=1.70; ratio_norm=1.00, ratio_squash=1.00, distinctive_hit=True, all_tokens=True
Staffordshire Bull Terrier     -> n02093256-Staffordshire_bullterrier
  score=1.68; ratio_norm=0.98, ratio_squash=0.71, distinctive_hit=True, all_tokens=True
------------

In [ ]:
# ============================================================
# 8. Sample unrestricted breed folders
# ============================================================
import random

if not SKIP_BUILD:
    unrestricted_pool = [f for f in all_folders if f not in restricted_folders]
    rng = random.Random(SEED)
    unrestricted_folders = sorted(rng.sample(unrestricted_pool, N_UNRESTRICTED))

    print("Sampled unrestricted folders:", len(unrestricted_folders))
    for folder in unrestricted_folders:
        print(" -", folder)
else:
    print("Skipped: finished split already exists.")


Sampled unrestricted folders: 24
 - n02085782-Japanese_spaniel
 - n02086646-Blenheim_spaniel
 - n02088238-basset
 - n02089078-black-and-tan_coonhound
 - n02091831-Saluki
 - n02092002-Scottish_deerhound
 - n02092339-Weimaraner
 - n02093428-American_Staffordshire_terrier
 - n02094258-Norwich_terrier
 - n02095570-Lakeland_terrier
 - n02097209-standard_schnauzer
 - n02098413-Lhasa
 - n02099429-curly-coated_retriever
 - n02100236-German_short-haired_pointer
 - n02100877-Irish_setter
 - n02101388-Brittany_spaniel
 - n02105412-kelpie
 - n02106030-collie
 - n02107574-Greater_Swiss_Mountain_dog
 - n02110627-affenpinscher
 - n02112137-chow
 - n02113023-Pembroke
 - n02115913-dhole
 - n02116738-African_hunting_dog


In [ ]:
# ============================================================
# 9. Build train/val/test split locally
# ============================================================
import hashlib
import shutil

def stable_seed(text: str) -> int:
    digest = hashlib.md5(text.encode("utf-8")).hexdigest()
    return int(digest[:8], 16)

def image_files(folder: Path) -> list[Path]:
    return sorted([
        p for p in folder.iterdir()
        if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
    ])

def split_one_breed(folder_name: str, label: str):
    src_dir = LOCAL_IMAGES / folder_name
    imgs = image_files(src_dir)

    rng = random.Random(SEED + stable_seed(folder_name))
    rng.shuffle(imgs)

    n = len(imgs)
    n_train = int(n * SPLIT[0])
    n_val = int(n * SPLIT[1])

    batches = {
        "train": imgs[:n_train],
        "val": imgs[n_train:n_train + n_val],
        "test": imgs[n_train + n_val:],
    }

    breed_slug = re.sub(r"^n\d+[-_]", "", folder_name).lower()

    counts = {}
    for split_name, batch in batches.items():
        dst_dir = LOCAL_SPLIT / split_name / label
        dst_dir.mkdir(parents=True, exist_ok=True)

        for src in batch:
            dst_name = f"{breed_slug}_{src.name}"
            shutil.copy2(src, dst_dir / dst_name)

        counts[split_name] = len(batch)

    return counts

if not SKIP_BUILD:
    if LOCAL_SPLIT.exists():
        shutil.rmtree(LOCAL_SPLIT)
    LOCAL_SPLIT.mkdir(parents=True, exist_ok=True)

    all_selected = [
        (folder, "restricted") for folder in restricted_folders
    ] + [
        (folder, "unrestricted") for folder in unrestricted_folders
    ]

    totals = {
        "train": {"restricted": 0, "unrestricted": 0},
        "val": {"restricted": 0, "unrestricted": 0},
        "test": {"restricted": 0, "unrestricted": 0},
    }

    print("Building local split:")
    print("-" * 70)

    for folder_name, label in all_selected:
        counts = split_one_breed(folder_name, label)
        for split_name, n in counts.items():
            totals[split_name][label] += n
        print(f"{label:12s} {folder_name:35s} "
              f"train={counts['train']:3d}, val={counts['val']:3d}, test={counts['test']:3d}")

    print("-" * 70)
    print("Local split totals:")
    for split_name in ["train", "val", "test"]:
        print(split_name, totals[split_name])
else:
    print("Skipped: finished split already exists.")


Building local split:
----------------------------------------------------------------------
restricted   n02087394-Rhodesian_ridgeback       train=103, val= 34, test= 35
restricted   n02093256-Staffordshire_bullterrier train= 93, val= 31, test= 31
restricted   n02106550-Rottweiler                train= 91, val= 30, test= 31
restricted   n02106662-German_shepherd           train= 91, val= 30, test= 31
restricted   n02107142-Doberman                  train= 90, val= 30, test= 30
restricted   n02108422-bull_mastiff              train= 93, val= 31, test= 32
unrestricted n02085782-Japanese_spaniel          train=111, val= 37, test= 37
unrestricted n02086646-Blenheim_spaniel          train=112, val= 37, test= 39
unrestricted n02088238-basset                    train=105, val= 35, test= 35
unrestricted n02089078-black-and-tan_coonhound   train= 95, val= 31, test= 33
unrestricted n02091831-Saluki                    train=120, val= 40, test= 40
unrestricted n02092002-Scottish_deerhound        

In [ ]:
# ============================================================
# 10. Copy finished split to Drive
# ============================================================
if not SKIP_BUILD:
    print("Copying finished split to Drive.")
    print("This still writes many files, but only the selected final dataset, not all 120 breeds.")

    for drive_folder in [DRIVE_TRAIN, DRIVE_VAL, DRIVE_TEST]:
        if drive_folder.exists():
            shutil.rmtree(drive_folder)

    for split_name in ["train", "val", "test"]:
        src = LOCAL_SPLIT / split_name
        dst = DRIVE_DATA / split_name
        shutil.copytree(src, dst)
        print(f"Copied {src} -> {dst}")

    print("Finished copying split to Drive.")
else:
    print("Skipped: finished split already exists.")


Copying finished split to Drive.
This still writes many files, but only the selected final dataset, not all 120 breeds.
Copied /content/restricted_dogs_work/split/train -> /content/drive/MyDrive/restricted-dog-computer-vision/data/train
Copied /content/restricted_dogs_work/split/val -> /content/drive/MyDrive/restricted-dog-computer-vision/data/val
Copied /content/restricted_dogs_work/split/test -> /content/drive/MyDrive/restricted-dog-computer-vision/data/test
Finished copying split to Drive.


In [ ]:
# ============================================================
# 11. Final verification
# ============================================================
def verify_split():
    ok = True
    print("=" * 70)
    print("FINAL DATASET VERIFICATION")
    print("=" * 70)

    for split_name, split_dir in [
        ("train", DRIVE_TRAIN),
        ("val", DRIVE_VAL),
        ("test", DRIVE_TEST),
    ]:
        for cls in CLASS_NAMES:
            folder = split_dir / cls
            n = count_images(folder)
            exists = folder.exists()
            marker = "OK" if exists and n > 0 else "MISSING/EMPTY"
            print(f"{marker:14s} {split_name:5s} / {cls:13s}: {n:5d} images")
            if not exists or n == 0:
                ok = False

    print("=" * 70)
    if not ok:
        raise RuntimeError("Dataset split is incomplete.")
    print("All checks passed.")
    print("Next notebook: 2_ModelZoo_Colab should read from:")
    print(DRIVE_DATA)

verify_split()


FINAL DATASET VERIFICATION
OK             train / unrestricted :  2462 images
OK             train / restricted   :   561 images
OK             val   / unrestricted :   816 images
OK             val   / restricted   :   186 images
OK             test  / unrestricted :   838 images
OK             test  / restricted   :   190 images
All checks passed.
Next notebook: 2_ModelZoo_Colab should read from:
/content/drive/MyDrive/restricted-dog-computer-vision/data


## Notes for later notebooks

Use this fixed class order everywhere:

```python
CLASS_NAMES = ["unrestricted", "restricted"]
```

When using `tf.keras.utils.image_dataset_from_directory`, pass:

```python
class_names=CLASS_NAMES
label_mode="binary"
```

This pins:

- `0 = unrestricted`
- `1 = restricted`

That matters for recall, precision, confusion matrices, threshold tuning, and Grad-CAM interpretation.
